# v1: BrowseComp-Plus Deep Research Agent 初版

这个 notebook 在 `agent_vllm.ipynb` 的单步 baseline 和 `agent_vllm_weather.ipynb` 的工具调用示例基础上，搭建一个初版多轮检索 agent。

本文件只定义代码，不在本地执行。运行前需要在服务器上准备好 BM25 索引并启动 vLLM OpenAI-compatible endpoint。

## 1. 配置与项目导入

v1 继续使用课程项目已有的 `agent/` 代码：BM25 检索器、vLLM 客户端、数据读取和评测函数。

In [ ]:
from pathlib import Path
import json
import re
import sys
from typing import Any, Dict, List, Optional

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.dataset_utils import load_jsonl
from agent.eval import run_evaluation
from agent.tools import build_searcher, get_agent_tool_specs_and_registry
from agent.vllm_client import VLLMClient

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"
API_KEY = "dummy"

BM25_INDEX_PATH = str(project_root / "indexes" / "browsecomp_plus_bm25.sqlite")
HARD50_PATH = str(project_root / "browsecomp_plus_hard50.jsonl")
SUBMISSION_PATH = str(project_root / "runs" / "v1_submission.jsonl")

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=BM25_INDEX_PATH)
base_tool_specs, base_tool_registry = get_agent_tool_specs_and_registry(
    searcher=searcher,
    k=5,
    snippet_max_chars=1200,
)


## 2. 补充局部文档查找工具

`agent/tools.py` 已提供 `search` 和 `get_document`。这里在 notebook 内增加一个轻量 `find_in_document`，只在已给定 docid 的文档文本中按关键词返回窗口片段，不引入外部知识库，也不替换检索器。

In [ ]:
def find_in_document(
    docid: str,
    keyword: str,
    window_chars: int = 700,
    max_matches: int = 5,
) -> Dict[str, Any]:
    doc = searcher.get_document(docid)
    if doc is None:
        return {"docid": docid, "keyword": keyword, "matches": [], "error": "document not found"}

    text = doc.get("text", "")
    keyword = keyword.strip()
    if not keyword:
        return {"docid": docid, "keyword": keyword, "matches": [], "error": "empty keyword"}

    lowered_text = text.lower()
    lowered_keyword = keyword.lower()
    matches = []
    start = 0
    while len(matches) < max_matches:
        pos = lowered_text.find(lowered_keyword, start)
        if pos < 0:
            break
        left = max(0, pos - window_chars // 2)
        right = min(len(text), pos + len(keyword) + window_chars // 2)
        matches.append({"start": pos, "snippet": text[left:right].strip()})
        start = pos + len(keyword)

    return {
        "docid": docid,
        "url": doc.get("url", ""),
        "keyword": keyword,
        "matches": matches,
        "num_matches": len(matches),
    }


find_tool_spec = {
    "type": "function",
    "function": {
        "name": "find_in_document",
        "description": "Find keyword occurrences inside a known BrowseComp-Plus document and return local snippets.",
        "parameters": {
            "type": "object",
            "properties": {
                "docid": {"type": "string", "description": "Document id to inspect."},
                "keyword": {"type": "string", "description": "Keyword or phrase to find."},
            },
            "required": ["docid", "keyword"],
        },
    },
}

tool_specs = list(base_tool_specs) + [find_tool_spec]
tool_registry = dict(base_tool_registry)
tool_registry["find_in_document"] = find_in_document


## 3. 工具执行与状态辅助函数

这些函数负责解析 tool call、执行本地工具、压缩过长结果、维护已搜索 query 和已见 docid，并从最终回答中提取 `Exact Answer`。

In [ ]:
def canonical_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).strip().lower())


def truncate_text(text: str, max_chars: int) -> str:
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "... [truncated]"


def parse_tool_arguments(raw_arguments: Any) -> Dict[str, Any]:
    if isinstance(raw_arguments, dict):
        return raw_arguments
    if raw_arguments in (None, ""):
        return {}
    try:
        parsed = json.loads(raw_arguments)
    except json.JSONDecodeError:
        return {"_argument_parse_error": str(raw_arguments)}
    return parsed if isinstance(parsed, dict) else {"_argument_parse_error": str(raw_arguments)}


def normalize_tool_result(tool_name: str, result: Any, max_doc_chars: int = 3500) -> Any:
    if tool_name == "get_document" and isinstance(result, dict):
        normalized = dict(result)
        if "text" in normalized:
            normalized["text"] = truncate_text(normalized["text"], max_doc_chars)
            normalized["text_is_truncated"] = len(str(result.get("text", ""))) > max_doc_chars
        return normalized

    if tool_name == "search" and isinstance(result, list):
        compact = []
        for item in result:
            if not isinstance(item, dict):
                compact.append(item)
                continue
            compact.append(
                {
                    "docid": item.get("docid", ""),
                    "score": item.get("score", 0),
                    "url": item.get("url", ""),
                    "snippet": truncate_text(item.get("snippet", ""), 1200),
                }
            )
        return compact

    return result


def execute_tool_call(tool_call: Dict[str, Any], registry: Dict[str, Any]) -> Dict[str, Any]:
    function = tool_call.get("function", {}) or {}
    name = function.get("name", "")
    arguments = parse_tool_arguments(function.get("arguments", "{}"))

    if "_argument_parse_error" in arguments:
        return {"ok": False, "tool_name": name, "arguments": arguments, "error": "invalid JSON arguments"}
    if name not in registry:
        return {"ok": False, "tool_name": name, "arguments": arguments, "error": "unknown tool"}

    try:
        raw_result = registry[name](**arguments)
        result = normalize_tool_result(name, raw_result)
        return {"ok": True, "tool_name": name, "arguments": arguments, "tool_result": result}
    except Exception as exc:
        return {"ok": False, "tool_name": name, "arguments": arguments, "error": repr(exc)}


def make_tool_message(tool_call: Dict[str, Any], executed: Dict[str, Any]) -> Dict[str, Any]:
    content = executed.get("tool_result") if executed.get("ok") else executed
    return {
        "role": "tool",
        "tool_call_id": tool_call.get("id", ""),
        "content": json.dumps(content, ensure_ascii=False),
    }


def initial_state(question: str) -> Dict[str, Any]:
    return {
        "question": question,
        "searched_query_keys": [],
        "searched_queries": [],
        "seen_docids": [],
        "tool_events": [],
        "rounds": [],
        "stale_rounds": 0,
    }


def summarize_tool_result(tool_name: str, result: Any) -> str:
    if tool_name == "search" and isinstance(result, list):
        docids = [str(item.get("docid", "")) for item in result if isinstance(item, dict)]
        return "search returned docids: " + ", ".join([d for d in docids if d][:8])
    if tool_name == "get_document" and isinstance(result, dict):
        return f"opened docid={result.get('docid', '')}, url={result.get('url', '')}"
    if tool_name == "find_in_document" and isinstance(result, dict):
        return f"found {result.get('num_matches', 0)} matches for {result.get('keyword', '')} in {result.get('docid', '')}"
    return truncate_text(json.dumps(result, ensure_ascii=False), 300)


def update_state_from_execution(state: Dict[str, Any], executed: Dict[str, Any], round_id: int) -> int:
    tool_name = executed.get("tool_name", "")
    arguments = executed.get("arguments", {}) or {}
    result = executed.get("tool_result")
    new_info = 0

    if tool_name == "search":
        query = arguments.get("query", "")
        query_key = canonical_text(query)
        if query_key and query_key not in state["searched_query_keys"]:
            state["searched_query_keys"].append(query_key)
            state["searched_queries"].append(query)
            new_info += 1
        if isinstance(result, list):
            for item in result:
                if isinstance(item, dict):
                    docid = str(item.get("docid", ""))
                    if docid and docid not in state["seen_docids"]:
                        state["seen_docids"].append(docid)
                        new_info += 1

    if tool_name in {"get_document", "find_in_document"}:
        docid = str(arguments.get("docid", ""))
        if docid and docid not in state["seen_docids"]:
            state["seen_docids"].append(docid)
            new_info += 1

    state["tool_events"].append(
        {
            "round_id": round_id,
            "tool_name": tool_name,
            "arguments": arguments,
            "ok": executed.get("ok", False),
            "summary": summarize_tool_result(tool_name, result),
        }
    )
    return new_info


def public_state_summary(state: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "searched_queries": state["searched_queries"][-10:],
        "seen_docids": state["seen_docids"][-25:],
        "recent_tool_events": state["tool_events"][-10:],
        "rounds": state["rounds"][-10:],
        "stale_rounds": state["stale_rounds"],
    }


def extract_exact_answer(text: str) -> str:
    text = str(text or "").strip()
    if not text:
        return ""
    match = re.search(r"(?im)^\s*Exact Answer\s*:\s*(.+?)\s*$", text)
    if match:
        return match.group(1).strip()
    match = re.search(r"(?im)^\s*Answer\s*:\s*(.+?)\s*$", text)
    return match.group(1).strip() if match else text


## 4. Deep Research Agent Loop

主循环允许模型多轮调用工具。完整轨迹保存在 `messages` 中；每轮实际调用模型时，会附加压缩后的研究状态，并限制最近消息数量，降低上下文无限增长的风险。

In [ ]:
SYSTEM_PROMPT = """
You are a Deep Research Agent for BrowseComp-Plus.
Answer complex questions using only evidence from the provided local tools.

Available tools:
- search(query): search the BrowseComp-Plus BM25 index.
- get_document(docid): open a retrieved document by docid.
- find_in_document(docid, keyword): locate a keyword inside a known document.

Behavior rules:
1. Use iterative search. If evidence is insufficient, reformulate the query and search again.
2. Prefer opening or inspecting promising documents before committing to an answer.
3. Track which queries and documents have already been tried. Avoid repeating the same search.
4. Do not use outside knowledge or unsupported guesses.
5. Stop searching only when the answer is supported by evidence, or when the loop controller asks you to finalize.

Final answer format:
Explanation: <brief evidence-based reasoning>
Evidence: <docids or snippets that support the answer>
Exact Answer: <short final answer only>
Confidence: <0-100>%
""".strip()


def build_model_messages(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    recent_message_limit: int = 12,
) -> List[Dict[str, Any]]:
    state_message = {
        "role": "user",
        "content": (
            "Current compressed research state. Use it to avoid repeated searches and to decide the next action.\n"
            + json.dumps(public_state_summary(state), ensure_ascii=False, indent=2)
        ),
    }
    recent = messages[2:]
    if len(recent) > recent_message_limit:
        recent = recent[-recent_message_limit:]
        while recent and recent[0].get("role") == "tool":
            recent = recent[1:]
    return [messages[0], messages[1], state_message] + recent


def response_to_assistant_message(response: Dict[str, Any]) -> tuple[Dict[str, Any], List[Dict[str, Any]]]:
    message = response["choices"][0]["message"]
    assistant_message = {"role": "assistant", "content": message.get("content") or ""}
    tool_calls = message.get("tool_calls") or []
    if tool_calls:
        assistant_message["tool_calls"] = tool_calls
    return assistant_message, tool_calls


def force_final_answer(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    reason: str,
    max_tokens: int = 900,
) -> str:
    final_instruction = {
        "role": "user",
        "content": (
            f"{reason}\n"
            "Do not call any more tools. Based only on the gathered evidence, produce the final answer now. "
            "If evidence is incomplete, give the best supported answer and lower the confidence."
        ),
    }
    response = client.simple_chat(
        model=MODEL_NAME,
        messages=build_model_messages(messages, state) + [final_instruction],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    assistant_message, _ = response_to_assistant_message(response)
    messages.append(final_instruction)
    messages.append({"role": "assistant", "content": assistant_message.get("content", "")})
    return assistant_message.get("content", "")


def run_deep_research_agent(
    question: str,
    query_id: Optional[str] = None,
    max_rounds: int = 6,
    stale_round_limit: int = 2,
    max_tokens: int = 1200,
    recent_message_limit: int = 12,
) -> Dict[str, Any]:
    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    state = initial_state(question)
    final_output = ""
    status = "max_rounds_reached"

    for round_id in range(1, max_rounds + 1):
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=build_model_messages(messages, state, recent_message_limit=recent_message_limit),
            temperature=0.0,
            max_tokens=max_tokens,
            tools=tool_specs,
            tool_choice="auto",
        )
        assistant_message, tool_calls = response_to_assistant_message(response)
        messages.append(assistant_message)
        state["rounds"].append(
            {
                "round_id": round_id,
                "assistant_content": truncate_text(assistant_message.get("content", ""), 400),
                "tool_call_count": len(tool_calls),
            }
        )

        if not tool_calls:
            final_output = assistant_message.get("content", "")
            status = "completed"
            break

        round_new_info = 0
        for tool_call in tool_calls:
            executed = execute_tool_call(tool_call, tool_registry)
            messages.append(make_tool_message(tool_call, executed))
            round_new_info += update_state_from_execution(state, executed, round_id)

        if round_new_info == 0:
            state["stale_rounds"] += 1
        else:
            state["stale_rounds"] = 0

        if state["stale_rounds"] >= stale_round_limit:
            status = "stopped_no_new_information"
            final_output = force_final_answer(
                messages,
                state,
                reason="The last tool calls did not add new queries or documents.",
            )
            break
    else:
        final_output = force_final_answer(
            messages,
            state,
            reason=f"The maximum number of research rounds ({max_rounds}) has been reached.",
        )

    predicted_answer = extract_exact_answer(final_output)
    return {
        "query_id": query_id,
        "status": status,
        "predicted_answer": predicted_answer,
        "final_output": final_output,
        "messages": messages,
        "state_summary": public_state_summary(state),
    }


## 5. 批量生成与评测辅助函数

服务器环境中可以调用 `generate_submission()` 生成统一 JSONL，再调用 `evaluate_submission()` 使用课程提供的评测脚本。

In [ ]:
def build_submission_record(row: Dict[str, Any], **agent_kwargs: Any) -> Dict[str, Any]:
    result = run_deep_research_agent(
        question=row["query"],
        query_id=str(row.get("query_id", "")),
        **agent_kwargs,
    )
    return {
        "query_id": str(row.get("query_id", "")),
        "predicted_answer": result["predicted_answer"],
        "status": result["status"],
        "messages": result["messages"],
        "state_summary": result["state_summary"],
    }


def generate_submission(
    dataset_path: str = HARD50_PATH,
    output_path: str = SUBMISSION_PATH,
    limit: Optional[int] = 50,
    **agent_kwargs: Any,
) -> List[Dict[str, Any]]:
    rows = load_jsonl(dataset_path, limit=limit)
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    records = []
    with output_file.open("w", encoding="utf-8") as fout:
        for index, row in enumerate(rows, start=1):
            print(f"[{index}/{len(rows)}] query_id={row.get('query_id', '')}")
            record = build_submission_record(row, **agent_kwargs)
            records.append(record)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(records)} records to {output_path}")
    return records


def evaluate_submission(
    submission_path: str = SUBMISSION_PATH,
    dataset_path: str = HARD50_PATH,
    output_path: str = str(project_root / "runs" / "v1_eval_results.jsonl"),
):
    return run_evaluation(
        submission_path=submission_path,
        dataset_path=dataset_path,
        model_name=MODEL_NAME,
        base_url=VLLM_BASE_URL,
        api_key=API_KEY,
        output_path=output_path,
        verbose=True,
    )


## 6. 运行示例

下面两行留给服务器环境执行。本地不具备算力和服务条件时不要运行。

```python
# records = generate_submission(limit=50, max_rounds=6, stale_round_limit=2)
# summary, details = evaluate_submission()
```